# 👁️ Real-Time Multimodal Binary Decision Model
### Zero-Shot Contrastive Visual Reasoning over Live Video Streams

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
[![PyTorch](https://img.shields.io/badge/PyTorch-2.0+-EE4C2C?logo=pytorch&logoColor=white)](https://pytorch.org/)
[![HuggingFace](https://img.shields.io/badge/HuggingFace-Transformers-yellow?logo=huggingface&logoColor=black)](https://huggingface.co/)

**Binary Decision Model** is a real-time multimodal visual question answering and decision-making system designed for live webcam feeds.
It uses explicit contrastive zero-shot pairs (`Positive Target vs Negative Target`) to perform arbitrary binary classification on the fly without retraining.


In [ ]:
# Install dependencies for Google Colab
!pip install -q transformers torch torchvision pillow opencv-python-headless ipywidgets mediapipe


## 🧪 1. Baseline Prototype: Single-Frame Contrastive CLIP
Real-time zero-shot binary classifier using OpenAI CLIP (`openai/clip-vit-large-patch14`) with contrastive prompt pairs.


In [ ]:
import io
import time
import base64
import torch
import numpy as np
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from IPython.display import display, HTML, JSON
from google.colab import output
import ipywidgets as widgets

# ============================================================
# 1. OPTIMIZED BINARY DECISION MODEL (High-Res + Contrastive Logic)
# ============================================================

class BinaryDecisionModel:
    pass

JEVModel = BinaryDecisionModel

class BinaryDecisionModel:
    def __init__(self, default_question: str = "clean shaven face vs bearded face"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"⚡ Loading High-Res CLIP (Large-Patch14) on {self.device}...")

        # Upgraded to patch14 for fine-grained detail (facial hair, textures, small objects)
        self.model = CLIPModel.from_pretrained("openai/clip-vit-large-patch14").to(self.device)
        self.processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")
        self.model.eval()

        self.text_features = None
        self.update_question(default_question)

    def _extract_tensor(self, out):
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            return out.pooler_output
        elif hasattr(out, "last_hidden_state"):
            return out.last_hidden_state[:, 0, :]
        return out

    def update_question(self, query: str):
        """Parses the 'vs' operator for explicit contrastive prompting."""
        self.question = query.strip() if query.strip() else "person"
        q = self.question.lower()

        # 🔥 EXPLICIT CONTRAST: The most powerful way to use CLIP
        if " vs " in q:
            pos_str, neg_str = q.split(" vs ", 1)
            self.positive_prompt = f"a photo clearly showing {pos_str.strip()}"
            self.negative_prompt = f"a photo clearly showing {neg_str.strip()}"
        else:
            # Fallback for simple objects
            cleaned = q.replace("is the person", "").replace("is there a", "").strip(" ?")
            self.positive_prompt = f"a photo of {cleaned}"
            self.negative_prompt = "a photo of a completely different object, scenery, or empty background"

        print(f"✅ Pos: '{self.positive_prompt}'")
        print(f"❌ Neg: '{self.negative_prompt}'")

        with torch.no_grad():
            inputs = self.processor(
                text=[self.positive_prompt, self.negative_prompt],
                return_tensors="pt",
                padding=True
            ).to(self.device)
            raw = self.model.get_text_features(**inputs)
            t_out = self._extract_tensor(raw)
            self.text_features = t_out / t_out.norm(dim=-1, keepdim=True)

    @torch.no_grad()
    def predict(self, pil_image: Image.Image) -> dict:
        t0 = time.time()

        inputs = self.processor(images=pil_image, return_tensors="pt").to(self.device)
        raw_img = self.model.get_image_features(**inputs)
        img_feats = self._extract_tensor(raw_img)
        img_feats = img_feats / img_feats.norm(dim=-1, keepdim=True)

        logits = (img_feats @ self.text_features.T) * self.model.logit_scale.exp()
        probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

        yes_prob = float(probs[0])
        no_prob = float(probs[1])
        latency_ms = (time.time() - t0) * 1000

        return {
            "yes_pct": round(yes_prob * 100, 1),
            "no_pct": round(no_prob * 100, 1),
            "is_yes": bool(yes_prob >= 0.5),
            "latency": round(latency_ms, 1)
        }

jev_classifier = JEVModel()

# ============================================================
# 2. PYTHON CALLBACK
# ============================================================

def colab_process_frame(data_url: str):
    try:
        header, encoded = data_url.split(",", 1)
        img_bytes = base64.b64decode(encoded)
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")

        pred = jev_classifier.predict(image)
        return JSON({
            "status": "ok",
            "yes_pct": pred["yes_pct"],
            "no_pct": pred["no_pct"],
            "is_yes": pred["is_yes"],
            "latency": pred["latency"],
            "question": jev_classifier.question
        })
    except Exception as e:
        return JSON({"status": "error", "error": str(e)})

output.register_callback("colab_process_frame", colab_process_frame)

# ============================================================
# 3. INTERACTIVE INPUT BAR & UI
# ============================================================

text_input = widgets.Text(
    value="clean shaven face vs bearded face",
    placeholder="Use 'vs' for best results (e.g., cat vs dog)",
    description="Query:",
    layout=widgets.Layout(width="500px")
)

submit_btn = widgets.Button(
    description="Update Query",
    button_style="info",
    icon="check"
)

def apply_query(_=None):
    val = text_input.value.strip()
    if val:
        print("\n--- Updating Target ---")
        jev_classifier.update_question(val)

text_input.on_submit(apply_query)
submit_btn.on_click(apply_query)

display(widgets.HBox([text_input, submit_btn]))

# Video stream and responsive Canvas overlay
stream_ui = HTML("""
<div style="font-family: monospace; max-width: 660px; margin-top: 10px;">
  <div style="position: relative; width: 640px; height: 480px; background: #000; border-radius: 8px; overflow: hidden;">
    <video id="webcam" autoplay playsinline muted style="width: 640px; height: 480px; object-fit: cover;"></video>
    <canvas id="hud" width="640" height="480" style="position: absolute; top: 0; left: 0; pointer-events: none;"></canvas>
  </div>
  <div style="margin-top: 8px;">
    <button id="toggleBtn" style="padding: 8px 16px; font-weight: bold; background: #e53935; color: #fff; border: none; border-radius: 4px; cursor: pointer;">
      Pause Stream
    </button>
  </div>
</div>

<script>
(async function() {
  const video = document.getElementById('webcam');
  const canvas = document.getElementById('hud');
  const ctx = canvas.getContext('2d');
  const toggleBtn = document.getElementById('toggleBtn');

  const captureCanvas = document.createElement('canvas');
  captureCanvas.width = 336; // Increased capture res slightly for patch14
  captureCanvas.height = 336;
  const captureCtx = captureCanvas.getContext('2d');

  let active = true;
  let inFlight = false;
  let state = {
    status: "initializing",
    question: "Connecting...",
    yes_pct: 0,
    no_pct: 0,
    is_yes: false,
    latency: 0
  };

  toggleBtn.onclick = () => {
    active = !active;
    toggleBtn.textContent = active ? "Pause Stream" : "Resume Stream";
    toggleBtn.style.background = active ? "#e53935" : "#43a047";
    if (active) requestAnimationFrame(loop);
  };

  try {
    const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 }, audio: false });
    video.srcObject = stream;
    await video.play();
  } catch (err) {
    alert("Camera access denied or unavailable: " + err);
    return;
  }

  function renderHUD() {
    ctx.clearRect(0, 0, canvas.width, canvas.height);

    ctx.fillStyle = 'rgba(15, 15, 20, 0.82)';
    ctx.fillRect(15, 15, 610, 100);

    ctx.strokeStyle = '#334155';
    ctx.lineWidth = 1;
    ctx.strokeRect(15, 15, 610, 100);

    if (state.status === "initializing") {
      ctx.fillStyle = '#facc15';
      ctx.font = '16px monospace';
      ctx.fillText("Connecting to Binary Decision Model backend...", 35, 65);
      return;
    }

    ctx.fillStyle = '#FFFFFF';
    ctx.font = '15px monospace';
    ctx.fillText("Q: " + state.question, 35, 42);

    ctx.font = 'bold 20px monospace';

    // Determine which side of the "vs" won for the label
    let label = "";
    if (state.question.includes(" vs ")) {
        const parts = state.question.split(" vs ");
        label = state.is_yes ? `DETECTED: ${parts[0]}` : `DETECTED: ${parts[1]}`;
    } else {
        label = state.is_yes ? "DETECTED: YES" : "DETECTED: NO";
    }

    ctx.fillStyle = state.is_yes ? '#22c55e' : '#3b82f6';
    const conf = state.is_yes ? state.yes_pct : state.no_pct;
    ctx.fillText(`${label.toUpperCase()} (${conf}%)`, 35, 75);

    ctx.font = '13px monospace';
    ctx.fillStyle = '#94a3b8';
    ctx.fillText(`${state.latency} ms`, 530, 75);

    ctx.fillStyle = '#1e293b';
    ctx.fillRect(35, 90, 570, 8);

    ctx.fillStyle = state.is_yes ? '#22c55e' : '#3b82f6';
    ctx.fillRect(35, 90, (570 * (state.yes_pct / 100)), 8);
  }

  async function loop() {
    if (!active) return;

    if (!inFlight) {
      inFlight = true;
      // Crop to square for better ViT scaling
      const minDim = Math.min(video.videoWidth, video.videoHeight);
      const startX = (video.videoWidth - minDim) / 2;
      const startY = (video.videoHeight - minDim) / 2;

      captureCtx.drawImage(video, startX, startY, minDim, minDim, 0, 0, captureCanvas.width, captureCanvas.height);
      const dataUrl = captureCanvas.toDataURL('image/jpeg', 0.7);

      google.colab.kernel.invokeFunction('colab_process_frame', [dataUrl], {})
        .then(res => {
          if (res && res.data && res.data['application/json']) {
            state = res.data['application/json'];
          }
          inFlight = false;
        })
        .catch(err => {
          inFlight = false;
        });
    }

    renderHUD();
    requestAnimationFrame(loop);
  }

  requestAnimationFrame(loop);
})();
</script>
""")

display(stream_ui)

## 🚀 2. Binary Decision Model v2: Multi-Modal Routed Visual Q&A (CLIP + X-CLIP + FaceMesh/Audio)

Intelligent multi-model system that automatically routes queries based on semantics:
- **Static visual traits** (`clean shaven vs bearded face`, `wearing glasses vs no glasses`) ➔ **CLIP ViT-L/14** (EMA smoothed)
- **Action & motion** (`waving vs still`, `jumping vs standing`) ➔ **X-CLIP** (8-frame spatio-temporal video window)
- **Speech & talking** (`talking vs silent`) ➔ **FaceMesh lip variance + WebAudio mic energy**
- **Manual overrides**: Prefix query with `static: ...`, `video: ...`, or `talk: ...`


In [ ]:
# ============================================================
# Binary Decision Model v2: Routed live visual Q&A (CLIP + X-CLIP + FaceMesh/Audio)
#
# Query routing:
#   static query  ("bearded vs clean shaven")  -> CLIP ViT-L/14, EMA-smoothed
#   action query  ("waving vs still")          -> X-CLIP on an 8-frame window
#   talking query ("talking vs silent")        -> FaceMesh mouth motion (+ mic energy)
# Force a route with a prefix: "static: ...", "video: ...", "talk: ..."
# ============================================================

# %% Install
# Dependencies installed in setup cell

# %% Imports
import io
import re
import time
import base64
import threading
from collections import deque

import numpy as np
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel, XCLIPProcessor, XCLIPModel
from IPython.display import display, HTML, JSON
from google.colab import output
import ipywidgets as widgets

try:
    import mediapipe as mp
    HAS_MEDIAPIPE = hasattr(mp, "solutions") and hasattr(mp.solutions, "face_mesh")
    if not HAS_MEDIAPIPE:
        print("ℹ️ MediaPipe solutions not present. Speech/talking queries will route via X-CLIP.")
except Exception as e:
    print("⚠️ MediaPipe unavailable, talking queries will fall back to X-CLIP:", e)
    HAS_MEDIAPIPE = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


# %% Query parsing + routing
TALK_WORDS = ("talk", "speak", "silent", "silence", "mouth", "lips", "quiet")
MOTION_WORDS = ("wav", "clap", "jump", "nod", "shak", "danc", "walk", "run", "eat",
                "drink", "laugh", "yawn", "point", "gestur", "mov", "writ", "typ",
                "wink", "blink", "wak", "stretch", "brush", "wash")
# "-ing" words that describe a state, not an action: keep these on plain CLIP
STATIC_ING = {"wearing", "holding", "carrying", "having", "smiling", "frowning",
              "standing", "sitting", "looking", "facing", "showing"}


def parse_query(raw: str):
    """Returns (forced_mode, pos, neg). neg may be None."""
    q = raw.strip().lower()
    forced = None
    m = re.match(r"^(static|video|talk)\s*:\s*(.*)$", q)
    if m:
        forced = {"static": "clip", "video": "xclip", "talk": "talk"}[m.group(1)]
        q = m.group(2)
    if " vs " in q:
        pos, neg = [s.strip() for s in q.split(" vs ", 1)]
    else:
        pos, neg = q.strip(" ?"), None
    return forced, pos, neg


def route(forced, pos, neg):
    if forced:
        return forced
    text = f"{pos} {neg or ''}"
    if any(w in text for w in TALK_WORDS) and HAS_MEDIAPIPE:
        return "talk"
    words = set(re.findall(r"\b\w+\b", text))
    if any(w in text for w in MOTION_WORDS):
        return "xclip"
    if any(w.endswith("ing") and w not in STATIC_ING for w in words):
        return "xclip"
    return "clip"


# %% Detectors
class ClipDetector:
    """Static queries. CLIP ViT-L/14 with explicit positive/negative prompts + EMA."""

    def __init__(self):
        print(f"⚡ Loading CLIP ViT-L/14 on {DEVICE}...")
        name = "openai/clip-vit-large-patch14"
        self.model = CLIPModel.from_pretrained(name).to(DEVICE).eval()
        self.processor = CLIPProcessor.from_pretrained(name)
        self.text_feats = None
        self.ema = None
        self.alpha = 0.4

    @staticmethod
    def _tensor(out):
        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            return out.pooler_output
        if hasattr(out, "last_hidden_state"):
            return out.last_hidden_state[:, 0, :]
        return out

    def set_query(self, pos, neg):
        if neg is not None:
            self.pos_prompt = f"a photo clearly showing {pos}"
            self.neg_prompt = f"a photo clearly showing {neg}"
        else:
            cleaned = pos.replace("is the person", "").replace("is there a", "").strip(" ?")
            self.pos_prompt = f"a photo of {cleaned}"
            self.neg_prompt = "a photo of a completely different object, scenery, or empty background"
        print(f"   ✅ Pos: '{self.pos_prompt}'\n   ❌ Neg: '{self.neg_prompt}'")
        with torch.no_grad():
            inputs = self.processor(text=[self.pos_prompt, self.neg_prompt],
                                    return_tensors="pt", padding=True).to(DEVICE)
            t = self._tensor(self.model.get_text_features(**inputs))
            self.text_feats = t / t.norm(dim=-1, keepdim=True)
        self.ema = None

    @torch.no_grad()
    def predict(self, image: Image.Image):
        inputs = self.processor(images=image, return_tensors="pt").to(DEVICE)
        f = self._tensor(self.model.get_image_features(**inputs))
        f = f / f.norm(dim=-1, keepdim=True)
        logits = (f @ self.text_feats.T) * self.model.logit_scale.exp()
        p = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()
        yes = float(p[0])
        self.ema = yes if self.ema is None else self.alpha * yes + (1 - self.alpha) * self.ema
        return self.ema, ""


class XClipDetector:
    """Action queries. X-CLIP scores an 8-frame clip against the two prompts."""

    N_FRAMES = 8
    WINDOW_S = 3.0          # spread the 8 frames across the last ~3 s
    RUN_EVERY_S = 0.5       # don't re-run inference more often than this

    def __init__(self):
        print(f"⚡ Loading X-CLIP on {DEVICE}...")
        name = "microsoft/xclip-base-patch32"
        self.model = XCLIPModel.from_pretrained(name).to(DEVICE).eval()
        self.processor = XCLIPProcessor.from_pretrained(name)
        self.frames = deque()           # (timestamp, PIL image)
        self.prompts = None
        self.ema = None
        self.alpha = 0.5
        self._last_run = 0.0

    def set_query(self, pos, neg):
        neg = neg if neg else "doing nothing, standing still"
        self.prompts = [f"a person {pos}", f"a person {neg}"]
        print(f"   ✅ Pos: '{self.prompts[0]}'\n   ❌ Neg: '{self.prompts[1]}'")
        self.frames.clear()
        self.ema = None

    def add(self, ts, image):
        if not self.frames or ts - self.frames[-1][0] >= 0.15:
            self.frames.append((ts, image.resize((224, 224))))
        while self.frames and ts - self.frames[0][0] > self.WINDOW_S + 1.0:
            self.frames.popleft()

    def _sample(self, now):
        recent = [f for f in self.frames if now - f[0] <= self.WINDOW_S]
        if len(recent) < self.N_FRAMES:
            return None, len(recent)
        idx = np.linspace(0, len(recent) - 1, self.N_FRAMES).round().astype(int)
        return [recent[i][1] for i in idx], len(recent)

    @torch.no_grad()
    def predict(self, ts, image):
        self.add(ts, image)
        clip_frames, n = self._sample(ts)
        if clip_frames is None:
            return None, f"Collecting frames {n}/{self.N_FRAMES}..."
        if ts - self._last_run >= self.RUN_EVERY_S or self.ema is None:
            self._last_run = ts
            arr = [np.array(f) for f in clip_frames]
            inputs = self.processor(text=self.prompts, videos=arr,
                                    return_tensors="pt", padding=True).to(DEVICE)
            probs = self.model(**inputs).logits_per_video.softmax(dim=1)[0].cpu().numpy()
            yes = float(probs[0])
            self.ema = yes if self.ema is None else self.alpha * yes + (1 - self.alpha) * self.ema
        return self.ema, ""


class TalkDetector:
    """Mouth-motion variance (FaceMesh) fused with mic energy from the browser."""

    WINDOW_S = 2.0

    def __init__(self):
        print("⚡ Loading MediaPipe FaceMesh...")
        self.mesh = mp.solutions.face_mesh.FaceMesh(
            max_num_faces=1, refine_landmarks=True,
            min_detection_confidence=0.5, min_tracking_confidence=0.5)
        self.mar_hist = deque()      # (ts, mouth aspect ratio)
        self.audio_hist = deque()    # (ts, rms)
        self.mouth_thr = 0.02        # std of MAR that counts as "moving"
        self.audio_thr = 0.02        # RMS that counts as "sound"
        self.use_audio = True
        self.ema = None
        self.alpha = 0.4

    def set_query(self, pos, neg):
        # Which side of the "vs" is the talking side?
        self.talking_is_pos = any(w in pos for w in TALK_WORDS) or not any(w in (neg or "") for w in TALK_WORDS)
        # "silent"/"quiet" on the positive side means talking is the negative side
        if any(w in pos for w in ("silent", "silence", "quiet")):
            self.talking_is_pos = False
        self.mar_hist.clear()
        self.audio_hist.clear()
        self.ema = None
        print(f"   🗣️ Talking is the {'LEFT' if self.talking_is_pos else 'RIGHT'} side of the query")

    def _mar(self, image):
        res = self.mesh.process(np.array(image))
        if not res.multi_face_landmarks:
            return None
        lm = res.multi_face_landmarks[0].landmark
        d = lambda a, b: np.hypot(lm[a].x - lm[b].x, lm[a].y - lm[b].y)
        return d(13, 14) / (d(61, 291) + 1e-6)   # inner-lip gap / mouth width

    def predict(self, ts, image, audio_rms):
        mar = self._mar(image)
        if mar is not None:
            self.mar_hist.append((ts, mar))
        if audio_rms is not None:
            self.audio_hist.append((ts, audio_rms))
        for h in (self.mar_hist, self.audio_hist):
            while h and ts - h[0][0] > self.WINDOW_S:
                h.popleft()

        if mar is None and len(self.mar_hist) < 3:
            return None, "No face detected"
        vals = [v for _, v in self.mar_hist]
        if len(vals) < 5:
            return None, f"Collecting frames {len(vals)}/5..."

        std = float(np.std(vals))
        p_mouth = float(sigmoid(4.0 * (std / self.mouth_thr - 1.0)))

        note = f"mouth σ={std:.3f}"
        p = p_mouth
        if self.use_audio and len(self.audio_hist) >= 5:
            frac = np.mean([r > self.audio_thr for _, r in self.audio_hist])
            p_audio = float(min(1.0, frac / 0.4))
            p = 0.65 * p_mouth + 0.35 * p_audio
            note += f" | audio {frac * 100:.0f}%"

        self.ema = p if self.ema is None else self.alpha * p + (1 - self.alpha) * self.ema
        p_yes = self.ema if self.talking_is_pos else 1.0 - self.ema
        return p_yes, note


# %% Router / orchestrator
class BinaryDecisionRouter:
    def __init__(self, default_query="clean shaven face vs bearded face"):
        self.lock = threading.Lock()
        self.clip = ClipDetector()
        self.xclip = None            # lazy-loaded on first action query
        self.talk = TalkDetector() if HAS_MEDIAPIPE else None
        self.mode = "clip"
        self.set_query(default_query)

    def set_query(self, raw):
        with self.lock:
            raw = raw.strip() or "person"
            forced, pos, neg = parse_query(raw)
            mode = route(forced, pos, neg)
            if mode == "talk" and self.talk is None:
                mode = "xclip"
            if mode == "xclip" and self.xclip is None:
                self.xclip = XClipDetector()
            self.mode, self.pos, self.neg, self.question = mode, pos, neg, raw
            print(f"🔀 Route: {mode.upper()}  |  query: '{raw}'")
            {"clip": self.clip, "xclip": self.xclip, "talk": self.talk}[mode].set_query(pos, neg)

    def process(self, image, audio_rms):
        t0 = time.time()
        with self.lock:
            if self.mode == "clip":
                p_yes, note = self.clip.predict(image)
            elif self.mode == "xclip":
                p_yes, note = self.xclip.predict(t0, image)
            else:
                p_yes, note = self.talk.predict(t0, image, audio_rms)
            mode, pos, neg, question = self.mode, self.pos, self.neg, self.question

        latency = round((time.time() - t0) * 1000, 1)
        base = {"mode": mode.upper(), "question": question, "latency": latency, "note": note}
        if p_yes is None:
            return {**base, "status": "warming", "label": "", "yes_pct": 0, "no_pct": 0, "is_yes": False}

        is_yes = p_yes >= 0.5
        if neg is not None:
            label = pos if is_yes else neg
        else:
            label = "YES" if is_yes else "NO"
        return {**base, "status": "ok", "label": label,
                "yes_pct": round(p_yes * 100, 1), "no_pct": round((1 - p_yes) * 100, 1),
                "is_yes": bool(is_yes)}


jev = BinaryDecisionRouter()
BinaryDecisionRouterAlias = BinaryDecisionRouter
JEVRouter = BinaryDecisionRouter


# %% Python callback invoked from the browser
def colab_process_frame(data_url: str, audio_rms=None):
    try:
        _, encoded = data_url.split(",", 1)
        image = Image.open(io.BytesIO(base64.b64decode(encoded))).convert("RGB")
        rms = float(audio_rms) if audio_rms is not None and audio_rms >= 0 else None
        return JSON(jev.process(image, rms))
    except Exception as e:
        return JSON({"status": "error", "error": str(e), "mode": "ERR", "question": "",
                     "label": "", "yes_pct": 0, "no_pct": 0, "is_yes": False,
                     "latency": 0, "note": str(e)[:60]})


output.register_callback("colab_process_frame", colab_process_frame)


# %% Controls
text_input = widgets.Text(
    value="clean shaven face vs bearded face",
    placeholder="e.g. talking vs silent | waving vs still | cat vs dog",
    description="Query:", layout=widgets.Layout(width="520px"))
submit_btn = widgets.Button(description="Update Query", button_style="info", icon="check")

mouth_slider = widgets.FloatSlider(value=0.02, min=0.005, max=0.08, step=0.005,
                                   description="Mouth σ thr", readout_format=".3f",
                                   style={"description_width": "90px"})
audio_slider = widgets.FloatSlider(value=0.02, min=0.005, max=0.15, step=0.005,
                                   description="Audio thr", readout_format=".3f",
                                   style={"description_width": "90px"})
audio_box = widgets.Checkbox(value=True, description="Fuse mic audio (talking mode)")


def apply_query(_=None):
    val = text_input.value.strip()
    if val:
        print("\n--- Updating Target ---")
        jev.set_query(val)


def apply_params(_=None):
    if jev.talk is not None:
        jev.talk.mouth_thr = mouth_slider.value
        jev.talk.audio_thr = audio_slider.value
        jev.talk.use_audio = audio_box.value


text_input.on_submit(apply_query)
submit_btn.on_click(apply_query)
for w in (mouth_slider, audio_slider, audio_box):
    w.observe(apply_params, names="value")

display(widgets.VBox([
    widgets.HBox([text_input, submit_btn]),
    widgets.HBox([mouth_slider, audio_slider]),
    audio_box,
]))

# %% Video stream + HUD
stream_ui = HTML("""
<div style="font-family: monospace; max-width: 660px; margin-top: 10px;">
  <div style="position: relative; width: 640px; height: 480px; background: #000; border-radius: 8px; overflow: hidden;">
    <video id="webcam" autoplay playsinline muted style="width: 640px; height: 480px; object-fit: cover;"></video>
    <canvas id="hud" width="640" height="480" style="position: absolute; top: 0; left: 0; pointer-events: none;"></canvas>
  </div>
  <div style="margin-top: 8px;">
    <button id="toggleBtn" style="padding: 8px 16px; font-weight: bold; background: #e53935; color: #fff; border: none; border-radius: 4px; cursor: pointer;">
      Pause Stream
    </button>
  </div>
</div>

<script>
(async function() {
  const video = document.getElementById('webcam');
  const canvas = document.getElementById('hud');
  const ctx = canvas.getContext('2d');
  const toggleBtn = document.getElementById('toggleBtn');

  const captureCanvas = document.createElement('canvas');
  captureCanvas.width = 336;
  captureCanvas.height = 336;
  const captureCtx = captureCanvas.getContext('2d');

  let active = true;
  let inFlight = false;
  let state = { status: "initializing", mode: "", question: "Connecting...", label: "",
                yes_pct: 0, no_pct: 0, is_yes: false, latency: 0, note: "" };

  toggleBtn.onclick = () => {
    active = !active;
    toggleBtn.textContent = active ? "Pause Stream" : "Resume Stream";
    toggleBtn.style.background = active ? "#e53935" : "#43a047";
    if (active) requestAnimationFrame(loop);
  };

  // Camera (+ microphone if allowed; audio is optional)
  let analyser = null, audioBuf = null;
  try {
    const stream = await navigator.mediaDevices.getUserMedia({
      video: { width: 640, height: 480 }, audio: true });
    video.srcObject = stream;
    try {
      const AC = window.AudioContext || window.webkitAudioContext;
      const actx = new AC();
      const src = actx.createMediaStreamSource(stream);
      analyser = actx.createAnalyser();
      analyser.fftSize = 1024;
      src.connect(analyser);           // not connected to destination: no echo
      audioBuf = new Uint8Array(analyser.fftSize);
    } catch (e) { analyser = null; }
  } catch (err) {
    try {
      const stream = await navigator.mediaDevices.getUserMedia({ video: { width: 640, height: 480 }, audio: false });
      video.srcObject = stream;
    } catch (err2) {
      alert("Camera access denied or unavailable: " + err2);
      return;
    }
  }
  await video.play();

  function audioRms() {
    if (!analyser) return -1;          // -1 = no audio available
    analyser.getByteTimeDomainData(audioBuf);
    let sum = 0;
    for (let i = 0; i < audioBuf.length; i++) {
      const v = (audioBuf[i] - 128) / 128;
      sum += v * v;
    }
    return Math.sqrt(sum / audioBuf.length);
  }

  function renderHUD() {
    ctx.clearRect(0, 0, canvas.width, canvas.height);
    ctx.fillStyle = 'rgba(15, 15, 20, 0.82)';
    ctx.fillRect(15, 15, 610, 110);
    ctx.strokeStyle = '#334155';
    ctx.lineWidth = 1;
    ctx.strokeRect(15, 15, 610, 110);

    if (state.status === "initializing") {
      ctx.fillStyle = '#facc15';
      ctx.font = '16px monospace';
      ctx.fillText("Connecting to Binary Decision Model...", 35, 70);
      return;
    }

    ctx.fillStyle = '#FFFFFF';
    ctx.font = '15px monospace';
    ctx.fillText("Q: " + state.question, 35, 40);

    ctx.fillStyle = '#94a3b8';
    ctx.font = '12px monospace';
    ctx.fillText("[" + state.mode + "]  " + (state.note || ""), 35, 58);

    if (state.status === "warming" || state.status === "error") {
      ctx.fillStyle = state.status === "error" ? '#ef4444' : '#facc15';
      ctx.font = 'bold 18px monospace';
      ctx.fillText(state.status === "error" ? "ERROR" : "WARMING UP...", 35, 90);
      ctx.fillStyle = '#94a3b8';
      ctx.font = '13px monospace';
      ctx.fillText(state.latency + " ms", 530, 90);
      return;
    }

    ctx.font = 'bold 20px monospace';
    ctx.fillStyle = state.is_yes ? '#22c55e' : '#3b82f6';
    const conf = state.is_yes ? state.yes_pct : state.no_pct;
    ctx.fillText(("DETECTED: " + state.label).toUpperCase() + " (" + conf + "%)", 35, 90);

    ctx.font = '13px monospace';
    ctx.fillStyle = '#94a3b8';
    ctx.fillText(state.latency + " ms", 530, 90);

    ctx.fillStyle = '#1e293b';
    ctx.fillRect(35, 105, 570, 8);
    ctx.fillStyle = state.is_yes ? '#22c55e' : '#3b82f6';
    ctx.fillRect(35, 105, 570 * (state.yes_pct / 100), 8);
  }

  async function loop() {
    if (!active) return;
    if (!inFlight) {
      inFlight = true;
      const minDim = Math.min(video.videoWidth, video.videoHeight);
      const sx = (video.videoWidth - minDim) / 2;
      const sy = (video.videoHeight - minDim) / 2;
      captureCtx.drawImage(video, sx, sy, minDim, minDim, 0, 0, captureCanvas.width, captureCanvas.height);
      const dataUrl = captureCanvas.toDataURL('image/jpeg', 0.7);

      google.colab.kernel.invokeFunction('colab_process_frame', [dataUrl, audioRms()], {})
        .then(res => {
          if (res && res.data && res.data['application/json']) {
            state = res.data['application/json'];
          }
          inFlight = false;
        })
        .catch(() => { inFlight = false; });
    }
    renderHUD();
    requestAnimationFrame(loop);
  }
  requestAnimationFrame(loop);
})();
</script>
""")

display(stream_ui)